# Scanpy Extra: (For quick testing)

In [37]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

Processing plate: plate1


### Set Variables

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from scanpy_init_env import *
from scanpy_utils import *
from scanpy_gene_lists import *

# Set variables
resolutions = [0.1, 0.2, 0.3] 

### Pull out L2 ExN-DL-0 cell IDs

In [ ]:
# L2 clusters
logger, root_dir, sheets_dir, plate_path, scanpy_dir, report_dir = initialize_env(plate)
subcluster_dir = scanpy_dir + f'subclustering/'
logger.info("Loading L2 data ...")
adata_sub = sc.read(subcluster_dir + 'adata_ExN-DL_subclusters.h5ad')

# Filter cells belonging to the 'ExN-DL-0' cluster
target_cluster = 'ExN-DL-0'
cells_in_cluster = adata_sub.obs.index[adata_sub.obs['subcluster'] == target_cluster]

# Optional: Convert to list
cell_list = cells_in_cluster.tolist()

# Save to a text file, one cell ID per line
output_path = f'{subcluster_dir}ExN-DL-0_v1_cellIDs.txt'
logger.info("Saving data ...")
with open(output_path, 'w') as f:
    for cell_id in cell_list:
        f.write(cell_id + '\n')

print(f"Saved {len(cell_list)} cell IDs to {output_path}")

del adata_sub

### Load L1 cells and change IDs

In [ ]:
# Load your L1 AnnData
adata = sc.read(scanpy_dir + 'adata_clusters.h5ad')

logger.info("Setting cluster IDs ...")
# Set cluster names
cluster_anns = {
    
    '0': 'ExN-UL',
    '1': 'ExN-DL',
    '2': 'RG',
    '3': 'InN',
    '4': 'Endo-Peri',
    '5': 'OPC',
    '6': 'MG'
}

custom_palette = {
    'RG': '#FF5959',
    'ExN-UL': '#00B6EB',
    'InN': '#3CBB75FF',
    'ExN-DL': '#CEE5FD',
    'Endo-Peri': '#B200ED',
    'MG': '#F58231',
    'OPC': '#FDE725FF'
}

# Create a new column in adata.obs with cell type names
adata.obs['cell_type'] = adata.obs['leiden_harmony_0.1'].map(cluster_anns)

# Check the new column
print(adata.obs[['leiden_harmony_0.1', 'cell_type']].head())

# Plot UMAP with updated annotation
plt.figure(figsize=(4, 4), dpi=150)  
sc.pl.umap(
    adata,
    color='cell_type',
    palette=custom_palette,
    title='L1: Before ExN-DL-0 cells are reassigned',
    show=False  # we'll control plt.show() manually
)

plt.tight_layout()  # neat layout
plt.show()

# Update only those cells in cell_list, changing their annotation to 'ExN-UL'
adata.obs.loc[adata.obs.index.isin(cell_list), 'cell_type'] = 'ExN-UL'

# Plot UMAP with updated annotation
plt.figure(figsize=(4, 4), dpi=150)  
sc.pl.umap(
    adata,
    color='cell_type',
    palette=custom_palette,
    title='L1: After ExN-DL-0 cells are reassigned',
    show=False  # we'll control plt.show() manually
)

plt.tight_layout()  # neat layout
plt.show()

print(adata.obs.loc[adata.obs.index.isin(cell_list), ['leiden_harmony_0.1', 'cell_type']].head(10))

#output_l1_path = scanpy_dir + 'adata_clusters_updated.h5ad'
#logger.info(f"Saving updated L1 AnnData to {output_l1_path} ...")
#adata.write(output_l1_path)
